# **Week 5 Challenge Activity NeuroSymbolic AI**

# Prepare Data
Before building any model, data preprocessing is often applied to prepare the data. Data preprocessing is the method of analyzing, filtering, transforming, and encoding data so that a machine learning algorithm can understand and work with the processed output

In the following code, the required libraries are loaded. Note: Ensure that all of these libraries are installed on your system. If not, install any missing libraries first.

Also, check the data source file path. In the provided code, the path is set to the data location on my system. You will need to modify the path according to your data's location.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
from numpy import array, argmax
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import warnings
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import TensorBoard
from sklearn.model_selection import train_test_split
import gc
from tqdm import tqdm_notebook as tqdm
import time

In [ ]:
# Connect to your Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
ROOTDIR = "gdrive/MyDrive/Shared/ChallengeActivity/"
DATADIR = "gdrive/MyDrive/Shared/ChallengeActivity/Week5Image/Leaves/"
CATEGORIES = ["LLeaves","MLeaves","HLeaves"]
IMG_SIZE = 200

In [ ]:
for category in CATEGORIES:
    path = os.path.join(DATADIR, category) #path to the Leaves
    for img in os.listdir(path):
        img_array = Image.open(os.path.join(path,img))
        img_array = img_array.resize((IMG_SIZE, IMG_SIZE))
        print(np.asarray(img_array).shape)
        plt.imshow(img_array)
        plt.show()
        break
    break

# Create Training Data
Please use the image dataset provided to you as XAI folder. The code below trains for leaves, but there are training images in the folder that you can use to train for other objects that could be possibly there in image, i.e. mud, water or bottles.

The categories indicate the coverage of the object within the image. For example, LLeaves means low coverage of leaves in the image. In training, we had to do manual annotation and apply some heuristics to define the coverage level, for example there is 20% or less leaves in entire image then we classify that as LLeaves. Similarly, heuristic ranges are used for Medium (MLeaves) and high leavels (HLeaves).

In [ ]:
n = 1500 # to shorten training time
X = np.zeros((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
y = np.zeros(n, dtype=int)

In [ ]:
def create_training_data():
    i = 0
    pbar = tqdm(total=n)

    for category in CATEGORIES:
        j = 0
        path = os.path.join(DATADIR, category) #path to the Leaves
        class_num = CATEGORIES.index(category)
        for img in os.listdir(path):
            try:
                img_array = Image.open(os.path.join(path,img))
                new_array = img_array.resize((IMG_SIZE, IMG_SIZE))
                X[i] = np.asarray(new_array)
                y[i] = class_num
                i = i + 1
                j = j + 1
                if j > 500:
                  break
                if i % 100 == 0:
                    gc.collect() # release memory
                pbar.update()
            except Exception as e:
                pass

    pbar.close()
    return i
n = create_training_data()
X = X[:n] # remove error's space
y = y[:n] # remove error's space

In [ ]:
indices = np.arange(n)
np.random.shuffle(indices)

In [ ]:
X = X[indices]
y = y[indices]

# Model Training
You can notice the code has encoding. Encoding is applied to handle categorical variables. It simply creates additional features based on the number of unique values in the categorical feature. Every unique value in the category will be added as a feature.

In [ ]:
# One Hot encoding for the class attribuites
# define example
values = array(y)
# integer encode
label_encoder = LabelEncoder()
integer_encoded = label_encoder.fit_transform(values)
# binary encode
onehot_encoder = OneHotEncoder(sparse_output=False)
integer_encoded = integer_encoded.reshape(len(integer_encoded), 1)
onehot_encoded = onehot_encoder.fit_transform(integer_encoded)
# invert first example
inverted = label_encoder.inverse_transform([argmax(onehot_encoded[0, :])])

y = onehot_encoded
X = X/255.0 # Normalise to 0-1

print("X:", X[:10])
print("y:", y[:10])

Define your ML model use summary function to see the details of your model.

In [ ]:
dense_layers = [2]
layer_sizes = [32]
conv_layers = [3]

for dense_layer in dense_layers:
    for layer_size in layer_sizes:
        for conv_layer in conv_layers:
            NAME = "{}-conv-{}-nodes-{}-dense".format(conv_layer, layer_size, dense_layer, int(time.time()))
            print(NAME)

            model = Sequential()

            model.add(Conv2D(layer_size, (3,3), input_shape = X.shape[1:]))
            model.add(Activation("relu"))
            model.add(MaxPooling2D(pool_size=(2,2)))

            for l in range(conv_layer-1):
                model.add(Conv2D(layer_size, (3,3)))
                model.add(Activation("relu"))
                model.add(MaxPooling2D(pool_size=(2,2)))

            model.add(Flatten())
            for l in range(dense_layer):
                model.add(Dense(layer_size))
                model.add(Activation("relu"))

            #add dropout layer
            model.add(Dropout(0.3, input_shape=(3,)))
            model.add(Dense(3))
            model.add(Activation('softmax'))

            tensorboard = TensorBoard(log_dir="leaves_logs{}".format(NAME))

            model.compile(loss="categorical_crossentropy",#categorical change to!
                         optimizer='adam',
                         metrics=['accuracy'])

In [ ]:
model.summary()

Train your model using model.fit function. Here, for simplicity only 4 epoch has been applied. You can change into a larger number (>40) in the model tuning phase

In [ ]:
model.fit(X, y,
            batch_size=32,
            epochs=4,
            validation_split=0.1,
            callbacks=[tensorboard],
            verbose=2)

Save your model for later usage. You can save anyewhere in your system using path name.

In [ ]:
model.save(f'{ROOTDIR}Week5Image/leaves-detection.keras')

# Test Model
Load your test images and check their corresponding classes. For this case, the classes are "LowLeaves," "MediumLeaves," and "HighLeaves."
You will obtain the probability for each class's output, which is a crucial feature for our rulebase concerning explainability. Based on these probabilities, the appropriate rules will be chosen for classification from the ontology (shared separately).

In the below you can see that the probability of high level of leaves is close to 55%. You might see different values based on training epochs/iterations. You can apply the same code to detect other objects and its coverage too.

In [ ]:
CATEGORIES = ["LowLeaves","MediumLeaves","HighLeaves"]
numOfClasses = len(CATEGORIES)

def prepare(filepath):

    img_array = Image.open(filepath)
    img_array = img_array.resize((IMG_SIZE, IMG_SIZE))
    plt.imshow(img_array)
    plt.show()
    return np.asarray(img_array).reshape(1, IMG_SIZE, IMG_SIZE, 3)

prediction = model.predict([prepare(f'{ROOTDIR}Week5Image/Leaves/test-image-1.jpg')/255])
print(prediction[0])

# Can you enhance the accuracy?
For instance, consider conducting model hyperparameter tuning by testing various combinations of hyperparameters (such as batch size, number of epochs, and others) and evaluating their performance.
Image augmentation is a technique used to artificially enlarge a dataset, especially beneficial when working with limited samples. In deep learning, training on a small dataset can lead to overfitting, making augmentation a valuable approach.

In [ ]:
!pip install owlready2

In [ ]:
# Code Refer to https://www.kaggle.com/code/kooaslansefat/neurosymbolic-ai-flooding-ka?scriptVersionId=202650960
import owlready2
import networkx as nx


# Load the ontology
onto = owlready2.get_ontology(f'{ROOTDIR}Object_coverage.owl').load()

def map_prediction_to_ontology(prediction):
    # Assuming 'prediction' is a list or array containing the probabilities for each class
    # Example class names corresponding to the model's output
    classes = ["Leaves", "Mud", "Plastic_Bottle"]
    # Get the index of the class with the highest probability
    predicted_index = prediction.argmax()
    # Map the index to the corresponding class name
    detected_object = classes[predicted_index]
    return detected_object


def update_ontology_with_predictions(predictions):
    for prediction in predictions:
        # Map prediction to ontology concepts (e.g., 'Leaves', 'Mud')
        detected_object = map_prediction_to_ontology(prediction)

        # Check if the class exists in the ontology
        if hasattr(onto, detected_object):
            cls = getattr(onto, detected_object)
            # Create an instance of the class
            instance = cls(detected_object + "_instance")  # Unique instance name
            # Add relevant relationships or properties
            instance.is_a.append(onto.Drainage_Classification)
            print(f"Added instance: {instance}")
        else:
            print(f"Class '{detected_object}' not found in the ontology.")

# Example workflow
test_image_path = f'{ROOTDIR}Week5Image/Leaves/test-image-1.jpg'
prediction = model.predict([prepare(test_image_path) / 255])
update_ontology_with_predictions(prediction)

# Save the updated ontology
onto.save(file=f'{ROOTDIR}Week5Image/updated_ontology.owl', format='rdfxml')

In [ ]:
def visualize_ontology(ontology):
    # Create a directed graph
    G = nx.DiGraph()

    # Add classes and their relationships to the graph
    for cls in ontology.classes():
        G.add_node(cls.name, type='class', color='lightblue')
        # Add subclass relationships
        for subclass in cls.subclasses():
            G.add_edge(cls.name, subclass.name, relationship='subclass')
        # Add instances of each class
        for instance in cls.instances():
            G.add_node(instance.name, type='instance', color='lightgreen')
            G.add_edge(cls.name, instance.name, relationship='instance_of')

    # Add object properties (relationships) to the graph
    for prop in ontology.object_properties():
        for domain in prop.domain:
            for range_ in prop.range:
                G.add_edge(domain.name, range_.name, relationship=prop.name)

    # Visualize the graph with directional edges
    plt.figure(figsize=(14, 14))
    pos = nx.spring_layout(G, k=0.5, iterations=50)

    # Draw nodes with different colors based on their types
    node_colors = [data['color'] for _, data in G.nodes(data=True)]
    nx.draw(G, pos, with_labels=True, node_size=3000, node_color=node_colors, font_size=10, font_weight="bold", edge_color="gray", arrows=True, arrowstyle='-|>')

    # Add edge labels to show the relationship types
    edge_labels = nx.get_edge_attributes(G, 'relationship')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=8)

    plt.title("Ontology Visualization with Updated Predictions")
    plt.show()

# Visualize the updated ontology
visualize_ontology(onto)

In [ ]:
!pip install owlready2 -q

In [ ]:
import owlready2
from graphviz import Digraph
from IPython.display import Image

def visualize_ontology_with_graphviz(ontology, output_file='ontology_graph'):
    # Create a Graphviz Digraph object
    dot = Digraph(comment='Ontology Visualization', format='png')

    # Add ontology classes as nodes
    for cls in ontology.classes():
        dot.node(cls.name, cls.name, shape='box', color='lightblue2', style='filled')

        # Add subclass relationships
        for subclass in cls.subclasses():
            dot.edge(cls.name, subclass.name, label='subclass_of')

        # Add instances for each class
        for instance in cls.instances():
            dot.node(instance.name, instance.name, shape='ellipse', color='lightgreen', style='filled')
            dot.edge(cls.name, instance.name, label='instance_of')

    # Add object properties as relationships between classes
    for prop in ontology.object_properties():
        for domain in prop.domain:
            for range_ in prop.range:
                dot.edge(domain.name, range_.name, label=prop.name, color='red')

    # Render the output and save the file
    output_path = dot.render(output_file)
    print(f"Ontology visualization saved to {output_path}")

    # Open the image using the default image viewer (cross-platform approach)
    try:
        if os.name == 'nt':  # For Windows
            os.startfile(output_path)
        elif os.name == 'posix':  # For macOS and Linux
            os.system(f'open "{output_path}"' if 'darwin' in os.sys.platform else f'xdg-open "{output_path}"')
    except Exception as e:
        print(f"Could not open the file automatically. Please open {output_path} manually.")

# Visualize the ontology
visualize_ontology_with_graphviz(onto, output_file='updated_ontology_graph')

# Display the generated ontology graph
Image(filename='updated_ontology_graph.png')